In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_5_try_0.pickle

trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  2
me:  2
trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  2
me:  2
trying: ['create_dataframe_of_counts_16']
me:  4
trying: ['percentages_per_country_df']
me:  4
trying: ['base_dir_2019']
me:  1
trying: ['factor']
me:  1
trying: ['go']
me:  0
trying: ['directory']
me:  1
trying: ['px']


me:  0
trying: ['base_dir_2020']
me:  1
trying: ['responses_in_order']
me:  5
trying: ['sns']
me:  0
trying: ['responses_df_2017']
me:  1
trying: ['glob']
me:  0
trying: ['file_name_2019']
me:  1
trying: ['file_name_2022']
me:  1
trying: ['pd']
me:  0
trying: ['responses_df_2020']
me:  1
trying: ['s']
me:  5
trying: ['percentages_cell17']
me:  5
trying: ['file_name_2018']
me:  1
trying: ['responses_df_2021']
me:  1
trying: ['counts']
me:  5
trying: ['directory_cell8']
me:  1
trying: ['orientation_for_chart']
me:  5
trying: ['load_survey_data']
me:  1
trying: ['question_of_interest']
me:  5
trying: ['base_dir_2017']
me:  1
trying: ['percentages']
me:  5
trying: ['base_dir_2018']
me:  1
trying: ['base_dir_2022']
me:  1
trying: ['responses_df_2019']
me:  1
trying: ['file_name_2020']
me:  1
trying: ['responses_df_2018']
me:  1
trying: ['base_dir_2021']
me:  1
trying: ['replace_hyphen_with_en_dash']
me:  2
trying: ['file_name_2017']
me:  1
trying: ['responses_df_2019_cell10']


me:  2
trying: ['file_name_2021']
me:  1
trying: ['np']
me:  0
trying: ['warnings']
me:  0
trying: ['responses_df_2018_cell10']
me:  2
Declaring variable go
Declaring variable px
Declaring variable sns
Declaring variable glob
Declaring variable pd
Declaring variable np
Declaring variable warnings
Declaring variable base_dir_2019
Declaring variable factor
Declaring variable directory
Declaring variable base_dir_2020
Declaring variable responses_df_2017
Declaring variable file_name_2019
Declaring variable file_name_2022
Declaring variable responses_df_2020
Declaring variable file_name_2018
Declaring variable responses_df_2021
Declaring variable directory_cell8
Declaring variable load_survey_data
Declaring variable base_dir_2017
Declaring variable base_dir_2018
Declaring variable base_dir_2022
Declaring variable responses_df_2019
Declaring variable file_name_2020
Declaring variable responses_df_2018
Declaring variable base_dir_2021
Declaring variable file_name_2017
Declaring variable file

In [3]:
%%RecordEvent
%%time
### cell 6 ###

def combine_subset_of_data_from_multiple_years_18(question_of_interest, include_2017=False):
    """
    Returns a DataFrame with columns ['percentage', 'year', question_of_interest]
    by computing the percent share of each value in question_of_interest
    for each year’s DataFrame.
    """
    # Map each year to its DataFrame
    year_dfs = [
        ('2022', responses_df_2022_cell10),
        ('2021', responses_df_2021),
        ('2020', responses_df_2020),
        ('2019', responses_df_2019_cell10),
        ('2018', responses_df_2018_cell10)
    ]
    if include_2017:
        year_dfs.append(('2017', responses_df_2017))

    records = []
    for year, df in year_dfs:
        # compute normalized counts → percentages
        s = (
            df[question_of_interest]
              .value_counts(dropna=False, normalize=True)
              .mul(100)
              .round(1)
              .sort_index()
        )
        temp = s.to_frame(name='percentage')
        temp['year'] = year
        # use question_of_interest as the label for the category column
        temp[question_of_interest] = temp.index
        records.append(temp[['percentage', 'year', question_of_interest]])

    # concatenate all years, reset index
    return pd.concat(records, ignore_index=True)


# --- Standardize raw DataFrames (unchanged) ---

question_name_alternate = 'Country'
question_name_alternate_cell18 = 'CurrentJobTitleSelect'

responses_df_2022_cell10.rename(
    columns={'United Kingdom (UK)': 'United Kingdom of Great Britain and Northern Ireland'},
    inplace=True
)
responses_df_2022_cell10.replace(
    'United Kingdom (UK)',
    'United Kingdom of Great Britain and Northern Ireland',
    inplace=True
)
responses_df_2017[question_name_alternate].replace({
    'United States': 'United States of America',
    "People 's Republic of China": 'China',
    'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland'
}, inplace=True)
responses_df_2017.rename(
    columns={
        question_name_alternate: 'In which country do you currently reside?',
        question_name_alternate_cell18: (
            'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
        )
    },
    inplace=True
)

# limit to a subset of countries
subset_of_countries = [
    'Other', 'India', 'United States of America', 'Brazil', 'Nigeria',
    'Pakistan', 'Japan', 'China', 'Egypt', 'Indonesia', 'Mexico',
    'Turkey', 'Russia'
]
question_name = 'In which country do you currently reside?'
for df in [
    responses_df_2017,
    responses_df_2018_cell10,
    responses_df_2019_cell10,
    responses_df_2020,
    responses_df_2021,
    responses_df_2022_cell10
]:
    df.loc[~df[question_name].isin(subset_of_countries), question_name] = 'Other'

# --- Combine and post-process ---
country_df_combined = combine_subset_of_data_from_multiple_years_18(
    question_name,
    include_2017=True
)

country_df_combined_cell18 = country_df_combined.sort_values(
    by=['year', 'percentage'],
    ascending=True
)
# correct the UK label one final time
country_df_combined_cell18[question_name] = country_df_combined_cell18[question_name].replace(
    'United Kingdom of Great Britain and Northern Ireland',
    'United Kingdom'
)

country_df_combined_cell18.info()

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/interactiveshell.py:2362: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result = fn(*args, **kwargs)


<class 'pandas.core.frame.DataFrame'>
Index: 78 entries, 67 to 3
Data columns (total 3 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   percentage                                 78 non-null     float64
 1   year                                       78 non-null     object 
 2   In which country do you currently reside?  78 non-null     object 
dtypes: float64(1), object(2)
memory usage: 2.4+ KB
CPU times: user 49.1 ms, sys: 0 ns, total: 49.1 ms
Wall time: 48.9 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_6_try_1.pickle

migration speed (bps): 755849798.1303737


---------------------------
variables to migrate:
replace_hyphen_with_en_dash 144
file_name_2017 76
file_name_2021 81
np 72
warnings 72
go 72
combine_subset_of_data_from_multiple_years_18 144
px 72
sns 72
glob 72
pd 72
question_of_interest 90
percentages 958
s 174012
percentages_cell17 958
responses_in_order 184
percentages_per_country_df 4474
counts 958
create_dataframe_of_counts_16 144
orientation_for_chart 50
question_name_alternate 56
question_name_alternate_cell18 70
question_name 90
base_dir_2019 155
responses_df_2022_cell10 25591023
factor 24
country_df_combined 10554
directory 163
responses_df_2017 15720777
base_dir_2020 155
responses_df_2022 25591023
responses_df_2020 25589809
file_name_2019 78
country_df_combined_cell18 11050
file_name_2022 81
responses_df_2018_cell10 32144293
responses_df_2021 34352595
file_name_2018 76
responses_df_2019_cell10 16055537
df 25591023
directory_cell8 170
subset_of_countries 168
load_survey_data 144
base_dir_2017 155
base_dir_2018 155
responses_

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2021', 'responses_df_2018', 'responses_df_2019', 'responses_df_2020', 'responses_df_2022']
Intermediate variables ['file_name_2020', 'base_dir_2019', 'file_name_2022', 'factor', 'responses_df_2017', 'base_dir_2018', 'base_dir_2022', 'file_name_2021', 'base_dir_2017', 'directory', 'base_dir_2021', 'file_name_2019', 'file_name_2017', 'directory_cell8', 'base_dir_2020', 'file_name_2018']
Future variables ['responses_df_2017', 'responses_df_2022_cell10']
Modified dataframes
======= Cell 2 =======
Input variables ['responses_df_2022', 'responses_df_2018', 'responses_df_2019']
Active variables ['responses_df_2019_cell10', 'responses_df_2022', 'responses_df_2022_cell10']
Intermediate variables ['responses_df_2018_cell10']
Future variables ['responses_df_2018_cell10', 'responses_df_2021', 'respo

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_6_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[6], f)


In [7]:
opt_output = Out.get(4)

In [8]:
subset_of_countries_opt = subset_of_countries
title_for_x_axis_opt = title_for_x_axis
responses_df_2021_opt = responses_df_2021
responses_df_2020_opt = responses_df_2020
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_6.pickle
assert subset_of_countries_opt == subset_of_countries
assert title_for_x_axis_opt == title_for_x_axis

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


NameError: name 'title_for_x_axis' is not defined